# Technical Memory — Cloudflare D1 plugin

- **File:** technical_memory.ipynb
- **Description:** Running log of technical decisions, tradeoffs and gotchas for the Cloudflare D1 plugin.
- **Author:** Jose-Jorge HERNANDEZ
- **Company:** Parlee Conseiller, Inc.
- **Date:** 2026-10-09
- **Last edit date:** 2026-10-09
- **Version:** 1.0.0


## 2026-10-09 — Tabularis v0.27 compatibility pass (v0.1.4)

**Context.** The plugin was built against Tabularis ~v0.15. Upstream is now v0.27.0 and the plugin contract (`plugins/PLUGIN_GUIDE.md`, `src-tauri/src/plugins/*`) changed.

**Decisions / gotchas**
- *Wire shapes unchanged.* The guide's JSON examples (`constraint_name`, `referenced_table`, `columns[]`) differ from the host's real structs (`name`, `ref_table`, one index row per column). The host structs in `models.rs` are authoritative; the plugin already matched them, so nothing was changed there.
- *Error code matters.* The host falls back on `-32601` for optional RPCs (`get_connection_metadata`, `get_table_query_template`). The old `-32603 "not implemented"` only worked for message-matching fallbacks like `ping`.
- *Notifications.* `cancel` has no `id`; replying breaks the "no unexpected stdout lines" rule. Requests are handled serially, so real cancellation is not possible without a reader thread — deliberately not done (out of scope).
- *Two manifests.* Host prefers `.tabularium`; `manifest.json` stays only for pre-0.14 hosts. Both keep `name: "Cloudflare D1"` plus `id: "cloudflare-d1"`, because old hosts treat `name` as the identifier and the registry slug must not change.
- *No local Rust toolchain* on the dev workstation: tests run in CI (`cargo test` step added to the release workflow).

## 2026-10-09 — macOS builds (v0.1.5)

**Decision.** Build natively on GitHub-hosted macOS runners (`macos-14` for arm64, `macos-15-intel` for x64) instead of cross-compiling, so `cargo test` runs on each architecture. `macos-13` is retired, hence `macos-15-intel`.

**Gotchas.** Registry platform keys are `darwin-arm64` / `darwin-x64` (not `macos-*`), per `plugins/README.md` upstream. `rustls-tls` means no OpenSSL; `build.rs` only embeds the icon on Windows. Binaries are unsigned and no Mac was available to run them in Tabularis, so only build + unit tests are verified.

## 2026-10-09 — Connection-form help box (v0.1.6)

**Problem.** Tabularis labels connection fields generically (Host/Username/Password) and the manifest has no way to relabel them, but the plugin maps Username→Account ID and Password→API token.

**Decision.** Use the `connection-modal.extra_fields` UI slot to render an explanatory box. Credentials stay in the real Username/Password fields: the slot's opaque `extra` map would hold the token outside the keychain-backed password field, so custom fields were rejected.

**Gotcha / tradeoff.** The host executes the bundle with `new Function(React, ReactJSXRuntime, __TABULARIS_API__, src)` and reads `__tabularis_plugin__`, so a ~100-line hand-written IIFE using `React.createElement` works with no Node/Vite toolchain. Checked by loading it through the same `new Function` wrapper in Node. The `driver` filter is done inside the component (`context.driver`) because the local schema doesn't list `driver` on ui_extensions entries. Colours use `currentColor`/opacity to follow any theme. English-only text (no locales file).

## 2026-10-09 — EXPLAIN, PRAGMA rows, triggers (v1.0.0)

**EXPLAIN.** Two wire shapes exist: raw (`engine/format/payload`, needs host >= v0.23.0, where Tabularis' built-in `sqlite-eqp-rows` parser would do the rendering) and the historical parsed plan (works on every host). Chose the parsed plan and a ~60-line tree builder so no `min_runtime_version` bump locks out older hosts. EQP rows have no single root (top-level rows have parent 0), so a synthetic 'Query Plan' node wraps them; depth is capped at 64 and self-parent rows skipped. `analyze` is ignored (SQLite has none).

**PRAGMA/EXPLAIN rows.** `is_select` also gated pagination (wrapping in `SELECT COUNT(*) FROM (...)`), and PRAGMA/EXPLAIN cannot be subqueries. Split into `is_select` (wrappable) and `returns_rows` (shown in the grid).

**Triggers.** SQLite keeps only the original SQL, so timing/event are parsed from the `CREATE TRIGGER` header (text before ` ON `, event searched only after the timing keyword so a trigger named `update_log` isn't misread; no timing keyword means BEFORE). Quoted names containing ` ON ` would confuse it; accepted edge case.

**Limits.** The Cloudflare connector wasn't authenticated this session, so live D1 behaviour wasn't tested; D1 docs confirm EXPLAIN is a supported read-only keyword. Version jumped to 1.0.0 as the first stable release (feature-complete for the host's driver contract; comments are N/A for D1).

## 2026-10-10 — Default Account ID + Spanish help text (v1.0.1)

**Settings.** Declared `settings` in the manifest; the host sends them in one `initialize` call before the first RPC (only when the manifest declares settings). The plugin keeps the value in a process-wide `Mutex<String>` and `resolve_account_id` prefers the connection's Username. Only the Account ID is a setting: Tabularis persists plugin settings in `config.json` (believed plaintext), whereas the Password field takes the keychain-backed path, so the token is intentionally NOT a setting.

**i18n.** The host loads `locales/<lang>.json` into an i18next namespace named after the plugin id and exposes `usePluginTranslation(id)` (just `useTranslation(id).t`). The hand-written bundle calls it before its early `return null` (hooks must not be conditional) and passes `defaultValue` so a missing key/locale/older host keeps English. Verified by running the bundle through the host's `new Function` wrapper with and without the hook.

**Open question.** Not verified whether the connection modal lets a connection be saved with Username blank; test in the app.